# Monoculture growth traits and competitive position

Monoculture maximum growth rate and lag time of the three core species, expressed relative to
the three-species mean in each environment, are related to each species' endpoint
competitive position q_i (its CLR coordinate) in the three-species communities.

Outputs: Figure 5, Figures S10-S12, Tables S12-S14.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple
import statsmodels.formula.api as smf

%run 00_setup_and_functions.ipynb

## Growth traits and competitive position
Kinetic analyses use 0, 2 and 20 ug/ml: at 200 ug/ml ancestral *A. caviae* and
*P. chlororaphis* barely grow, so their lag and rate estimates are not interpretable.
Each trait is expressed relative to the mean of the three species in the same environment and
standardized within history.

In [2]:
growth = load_growth()
TRAIT_DOSES = [0.0, 2.0, 20.0]
COMPETITION_TRAITS = ["max_growth_rate", "lag_time_h"]
growth_means = growth.groupby(["strain", "carbon_g_L", "streptomycin"], as_index=False)[GROWTH_TRAITS].mean()

core = sample_compositions.loc[sample_compositions.community.isin(["ANC_THREE", "EVO_THREE"])
                               & sample_compositions.streptomycin.isin(TRAIT_DOSES)].copy()
q = clr(core[CORE_THREE].to_numpy(float))
for j, sp in enumerate(CORE_THREE):
    core[f"q_{sp}"] = q[:, j]

rows = []
for community, strains in GROWTH_STRAIN_BY_COMMUNITY.items():
    for carbon in CARBON_VALUES:
        for dose in TRAIT_DOSES:
            for species in CORE_THREE:
                g = growth_means.loc[(growth_means.strain == strains[species]) & np.isclose(growth_means.carbon_g_L, carbon)
                                     & (growth_means.streptomycin == dose)]
                if len(g) != 1:
                    raise ValueError(f"Expected one growth mean for {strains[species]}, carbon {carbon}, dose {dose}")
                rows.append({"community": community, "carbon_g_L": carbon, "streptomycin": dose, "species": species,
                             **{t: float(g.iloc[0][t]) for t in COMPETITION_TRAITS}})
trait_condition = pd.DataFrame(rows)
for t in COMPETITION_TRAITS:
    trait_condition[f"relative_{t}"] = trait_condition[t] - trait_condition.groupby(["community", "carbon_g_L", "streptomycin"])[t].transform("mean")
    trait_condition[f"z_relative_{t}"] = trait_condition.groupby("community")[f"relative_{t}"].transform(lambda x: (x - x.mean()) / x.std(ddof=0))

competitive_position = pd.DataFrame([
    {"community": r.community, "history": "ancestral" if r.community == "ANC_THREE" else "resistance-evolved",
     "replicate": int(r.replicate), "carbon_g_L": float(r.carbon_g_L), "streptomycin": float(r.streptomycin),
     "species": sp, "q_endpoint": float(getattr(r, f"q_{sp}"))}
    for r in core.itertuples(index=False) for sp in CORE_THREE
]).merge(trait_condition, on=["community", "carbon_g_L", "streptomycin", "species"], how="left", validate="many_to_one")
if competitive_position[[f"z_relative_{t}" for t in COMPETITION_TRAITS]].isna().any().any():
    raise ValueError("Missing growth traits after merge")

# Condition means of q_i, plotted in Figures 5B,C and S11.
trait_condition_plot = competitive_position.groupby(["community", "carbon_g_L", "streptomycin", "species"], as_index=False).agg(
    q_endpoint=("q_endpoint", "mean"), z_relative_max_growth_rate=("z_relative_max_growth_rate", "first"),
    z_relative_lag_time_h=("z_relative_lag_time_h", "first"))

## Table S12: two-trait model per replicate landscape
q_i ~ z(relative maximum growth rate) + z(relative lag time) is fitted separately to each of
the eight landscapes per history (0, 2 and 20 ug/ml together). Mean partial slopes are tested
against zero by exact sign flips and between histories by exact allocation; Holm across the
two traits; bootstrap CIs resample landscapes.

In [3]:
MODEL = "q_endpoint ~ z_relative_max_growth_rate + z_relative_lag_time_h"
joint_slopes = pd.DataFrame([
    {"history": "ancestral" if c == "ANC_THREE" else "resistance-evolved", "replicate": int(r),
     "rate_slope": fit.params["z_relative_max_growth_rate"], "lag_slope": fit.params["z_relative_lag_time_h"]}
    for (c, r), dat in competitive_position.groupby(["community", "replicate"])
    for fit in [smf.ols(MODEL, data=dat).fit()]])

rows = []
for trait, col in [("Maximum growth rate", "rate_slope"), ("Lag time", "lag_slope")]:
    anc = joint_slopes.loc[joint_slopes.history == "ancestral", col].to_numpy(float)
    evo = joint_slopes.loc[joint_slopes.history == "resistance-evolved", col].to_numpy(float)
    anc_eff, anc_p = exact_signflip(anc)
    evo_eff, evo_p = exact_signflip(evo)
    diff, diff_p = exact_mean_difference_test(anc, evo)
    k = 1 if col == "rate_slope" else 2
    anc_ci = bootstrap_mean_ci(anc, seed=RANDOM_SEED + 6100 + k)
    evo_ci = bootstrap_mean_ci(evo, seed=RANDOM_SEED + 6200 + k)
    diff_ci = bootstrap_diff_ci(anc, evo, seed=RANDOM_SEED + 6300 + k)
    rows.append({"trait": trait, "ancestral_median_slope": np.median(anc), "evolved_median_slope": np.median(evo), "ancestral_slope": anc_eff, "ancestral_ci95_low": anc_ci[0], "ancestral_ci95_high": anc_ci[1],
                 "ancestral_exact_P": anc_p, "evolved_slope": evo_eff, "evolved_ci95_low": evo_ci[0], "evolved_ci95_high": evo_ci[1],
                 "evolved_exact_P": evo_p, "evolved_minus_ancestral": diff, "interaction_ci95_low": diff_ci[0],
                 "interaction_ci95_high": diff_ci[1], "interaction_exact_P": diff_p})
trait_tests = pd.DataFrame(rows)
for prefix in ["ancestral", "evolved", "interaction"]:
    trait_tests[f"{prefix}_Holm_P_across_two_traits"] = holm(trait_tests[f"{prefix}_exact_P"])
table_s12 = trait_tests[["trait", "ancestral_median_slope", "evolved_median_slope", "ancestral_slope", "ancestral_ci95_low", "ancestral_ci95_high", "ancestral_Holm_P_across_two_traits",
                         "evolved_slope", "evolved_ci95_low", "evolved_ci95_high", "evolved_Holm_P_across_two_traits",
                         "evolved_minus_ancestral", "interaction_ci95_low", "interaction_ci95_high", "interaction_Holm_P_across_two_traits"]]
table_s12.to_csv(TABLE_DIR / "Table_S12_growth_trait_model.csv", index=False)
table_s12

,trait,ancestral_median_slope,evolved_median_slope,ancestral_slope,ancestral_ci95_low,ancestral_ci95_high,ancestral_Holm_P_across_two_traits,evolved_slope,evolved_ci95_low,evolved_ci95_high,evolved_Holm_P_across_two_traits,evolved_minus_ancestral,interaction_ci95_low,interaction_ci95_high,interaction_Holm_P_across_two_traits
0,Maximum growth rate,0.855984,0.71041,0.864756,0.839870,0.892674,0.015625,0.716994,0.703490,0.734074,0.015625,-0.147762,-0.179377,-0.117546,0.000311
1,Lag time,-0.420103,-0.57917,-0.399301,-0.440522,-0.352687,0.015625,-0.547062,-0.592865,-0.475314,0.015625,-0.147761,-0.218117,-0.065319,0.006371


## Tables S13 and S14: dose-resolved models
The same model fitted separately at each dose within each landscape (Holm across the two traits
at each dose), and the Shapley/LMG partition of its R2 between the two traits: each trait's
contribution is its mean incremental R2 over the two orders of entry.

In [4]:
dose_rows, partition_rows = [], []
for (c, r, dose), dat in competitive_position.groupby(["community", "replicate", "streptomycin"]):
    history = "ancestral" if c == "ANC_THREE" else "resistance-evolved"
    fit = smf.ols(MODEL, data=dat).fit()
    dose_rows.append({"history": history, "replicate": int(r), "streptomycin": float(dose),
                      "rate_slope": fit.params["z_relative_max_growth_rate"], "lag_slope": fit.params["z_relative_lag_time_h"]})
    r2_rate = smf.ols("q_endpoint ~ z_relative_max_growth_rate", data=dat).fit().rsquared
    r2_lag = smf.ols("q_endpoint ~ z_relative_lag_time_h", data=dat).fit().rsquared
    r2 = fit.rsquared
    rate_part, lag_part = 0.5 * (r2_rate + (r2 - r2_lag)), 0.5 * (r2_lag + (r2 - r2_rate))
    partition_rows.append({"history": history, "replicate": int(r), "streptomycin": float(dose), "joint_R2": r2,
                           "rate_R2_contribution": rate_part, "lag_R2_contribution": lag_part,
                           "rate_share_of_explained": rate_part / r2 if r2 > 0 else np.nan,
                           "lag_share_of_explained": lag_part / r2 if r2 > 0 else np.nan})
dose_slopes = pd.DataFrame(dose_rows)

rows = []
for dose in TRAIT_DOSES:
    family = []
    for trait, col, k in [("Maximum growth rate", "rate_slope", 1), ("Lag time", "lag_slope", 2)]:
        anc = dose_slopes.loc[(dose_slopes.history == "ancestral") & (dose_slopes.streptomycin == dose), col].to_numpy(float)
        evo = dose_slopes.loc[(dose_slopes.history == "resistance-evolved") & (dose_slopes.streptomycin == dose), col].to_numpy(float)
        anc_eff, anc_p = exact_signflip(anc)
        evo_eff, evo_p = exact_signflip(evo)
        diff, diff_p = exact_mean_difference_test(anc, evo)
        anc_ci = bootstrap_mean_ci(anc, seed=RANDOM_SEED + 7000 + int(dose) * 10 + k)
        evo_ci = bootstrap_mean_ci(evo, seed=RANDOM_SEED + 7100 + int(dose) * 10 + k)
        diff_ci = bootstrap_diff_ci(anc, evo, seed=RANDOM_SEED + 7200 + int(dose) * 10 + k)
        family.append({"streptomycin": dose, "trait": trait, "ancestral_median_slope": np.median(anc), "evolved_median_slope": np.median(evo), "ancestral_slope": anc_eff, "ancestral_ci_low": anc_ci[0],
                       "ancestral_ci_high": anc_ci[1], "ancestral_P": anc_p, "evolved_slope": evo_eff, "evolved_ci_low": evo_ci[0],
                       "evolved_ci_high": evo_ci[1], "evolved_P": evo_p, "history_difference": diff,
                       "interaction_ci_low": diff_ci[0], "interaction_ci_high": diff_ci[1], "interaction_P": diff_p})
    for prefix in ["ancestral", "evolved", "interaction"]:
        for row, adj in zip(family, holm([f[f"{prefix}_P"] for f in family])):
            row[f"{prefix}_Holm_P"] = adj
    rows += family
table_s13 = pd.DataFrame(rows)[["streptomycin", "trait", "ancestral_median_slope", "evolved_median_slope", "ancestral_slope", "ancestral_ci_low", "ancestral_ci_high", "ancestral_Holm_P",
                                "evolved_slope", "evolved_ci_low", "evolved_ci_high", "evolved_Holm_P", "history_difference",
                                "interaction_ci_low", "interaction_ci_high", "interaction_Holm_P"]]
table_s13.to_csv(TABLE_DIR / "Table_S13_growth_trait_dose_models.csv", index=False)

table_s14 = pd.DataFrame(partition_rows).groupby(["history", "streptomycin"], as_index=False).agg(
    joint_R2=("joint_R2", "mean"), rate_R2_contribution=("rate_R2_contribution", "mean"),
    lag_R2_contribution=("lag_R2_contribution", "mean"), rate_share_of_explained=("rate_share_of_explained", "mean"),
    lag_share_of_explained=("lag_share_of_explained", "mean"), n_replicate_landscapes=("replicate", "nunique"))
table_s14.to_csv(TABLE_DIR / "Table_S14_growth_trait_variance_partition.csv", index=False)
table_s13

,streptomycin,trait,ancestral_median_slope,evolved_median_slope,ancestral_slope,ancestral_ci_low,ancestral_ci_high,ancestral_Holm_P,evolved_slope,evolved_ci_low,evolved_ci_high,evolved_Holm_P,history_difference,interaction_ci_low,interaction_ci_high,interaction_Holm_P
0,0.0,Maximum growth rate,1.018663,1.249759,1.027353,1.003400,1.053020,0.015625,1.253079,1.231460,1.275418,0.015625,0.225726,0.192282,0.259339,0.000311
1,0.0,Lag time,-0.769097,-0.850878,-0.735785,-0.789472,-0.646885,0.015625,-0.812714,-0.868106,-0.729726,0.015625,-0.076929,-0.186435,0.029549,0.257964
2,2.0,Maximum growth rate,1.134765,1.160522,1.127958,1.090518,1.159957,0.015625,1.158355,1.124501,1.193709,0.015625,0.030397,-0.017212,0.079732,0.275680
3,2.0,Lag time,-0.168724,-0.623208,-0.190806,-0.233502,-0.152129,0.015625,-0.579796,-0.639602,-0.476507,0.015625,-0.388990,-0.469881,-0.274454,0.000932
4,20.0,Maximum growth rate,0.428622,-0.080668,0.444655,0.399155,0.505063,0.015625,-0.077087,-0.118274,-0.037168,0.015625,-0.521741,-0.594304,-0.459014,0.000311
5,20.0,Lag time,-0.103316,-0.211184,-0.051874,-0.126180,0.063865,0.382812,-0.181244,-0.220184,-0.130672,0.015625,-0.129370,-0.255328,-0.034540,0.017716


## Figure 5
A, high-MIC share of non-focal abundance (Table S10); B,C, relative lag time and maximum growth rate versus competitive position, with median slopes across landscapes (Table S12); D, dose-resolved partition of R2 between the two traits (Table S14).

In [5]:
mic_landscape, _, _ = resistance_filtering()
fig = plt.figure(figsize=(183 * MM, 94 * MM))
gs = fig.add_gridspec(1, 4, width_ratios=[0.95, 1.00, 1.00, 1.40], wspace=0.48)

# A: share of non-focal abundance carried by high-MIC taxa (Table S10).
axA = fig.add_subplot(gs[0, 0]); panel_label(axA, "a")
for gi, (community, color) in enumerate([("ANC_SIXTEEN", ANCESTRAL_COLOR), ("EVO_SIXTEEN", EVOLVED_COLOR)]):
    pvt = (mic_landscape.loc[mic_landscape.community == community]
           .pivot(index="replicate", columns="streptomycin", values="high_MIC_share_excl_focal").reindex(columns=STREPTOMYCIN_VALUES))
    means, sems = [], []
    for xi, dose in enumerate(STREPTOMYCIN_VALUES):
        vals = 100 * pvt[dose].to_numpy(float)
        jitter = np.random.default_rng(4000 + gi * 100 + xi).uniform(-0.055, 0.055, len(vals))
        axA.scatter(xi + jitter, vals, s=10, facecolor="white", edgecolor=color, lw=0.65, alpha=0.88, zorder=3)
        means.append(vals.mean()); sems.append(vals.std(ddof=1) / np.sqrt(len(vals)))
    axA.plot(np.arange(4), means, color=color, lw=1.25, zorder=2)
    axA.errorbar(np.arange(4), means, yerr=1.96 * np.asarray(sems), fmt="o", mfc=color, mec=color, color=color, ms=4.0, lw=1.0, capsize=2, zorder=4)
axA.set_xticks(range(4)); axA.set_xticklabels([f"{x:g}" for x in STREPTOMYCIN_VALUES])
axA.set_ylim(0, 100); axA.set_xlabel(STREP_AXIS_LABEL)
axA.set_ylabel("Non-focal abundance in\nhigh-MIC taxa (%)")
axA.legend(handles=[
    Line2D([0], [0], marker="o", color=ANCESTRAL_COLOR, mfc=ANCESTRAL_COLOR, lw=1.0, label=r"Ancestral $\it{A.\ caviae}$"),
    Line2D([0], [0], marker="o", color=EVOLVED_COLOR, mfc=EVOLVED_COLOR, lw=1.0, label=r"Resistance-evolved $\it{A.\ caviae}$")],
    loc="upper left", frameon=False, fontsize=4.7)

# B, C: q_i against relative lag time and maximum growth rate; lines are the median partial
# slopes of the two-trait model across the N=8 replicate fits (Table S12), shading their
# bootstrap 95% CI.
for ax, panel, trait, xlabel, slope_col in [(fig.add_subplot(gs[0, 1]), "B", "lag_time_h", "Relative lag time (s.d.)", "lag_slope"),
                                            (fig.add_subplot(gs[0, 2]), "C", "max_growth_rate", "Relative maximum growth rate (s.d.)", "rate_slope")]:
    panel_label(ax, panel)
    ax.axhline(0, color="0.82", lw=0.6); ax.axvline(0, color="0.82", lw=0.6)
    xcol = f"z_relative_{trait}"
    for community, history, color in [("ANC_THREE", "ancestral", ANCESTRAL_COLOR), ("EVO_THREE", "resistance-evolved", EVOLVED_COLOR)]:
        dat = trait_condition_plot.loc[trait_condition_plot.community == community]
        ax.scatter(dat[xcol], dat["q_endpoint"], s=7.0, facecolor="white", edgecolor=color, lw=0.42, alpha=0.34, rasterized=True)
        slopes = joint_slopes.loc[joint_slopes.history == history, slope_col].to_numpy(float)
        xx = np.linspace(dat[xcol].min(), dat[xcol].max(), 100)
        boot = np.median(slopes[np.random.default_rng(RANDOM_SEED + 8700).integers(0, len(slopes), size=(10000, len(slopes)))], axis=1)
        lo, hi = np.quantile(boot[:, None] * xx[None, :], [0.025, 0.975], axis=0)
        ax.fill_between(xx, lo, hi, color=color, alpha=0.35, linewidth=0, zorder=3)
        ax.plot(xx, np.median(slopes) * xx, color=color, lw=0.9, zorder=4)
    ax.set_xlabel(xlabel)
    ax.tick_params(labelsize=5.7)
    if panel == "B":
        ax.set_ylabel(r"Competitive position $q_i$")
    else:
        ax.tick_params(axis="y", labelleft=False)

# D: dose-resolved partition of the joint R2 (Table S14).
axD = fig.add_subplot(gs[0, 3]); panel_label(axD, "d", dx=-30)
histories = [("ancestral", "Ancestral"), ("resistance-evolved", "Evolved")]
bar_h = 0.30
y_ticks, y_labels = [], []
for di, dose in enumerate(TRAIT_DOSES):
    for hi, (history, hlabel) in enumerate(histories):
        row = table_s14.loc[(table_s14.history == history) & (table_s14.streptomycin == dose)].iloc[0]
        y = di + (hi - 0.5) * (bar_h * 1.12)
        dark, light = (ANCESTRAL_COLOR, ANCESTRAL_LIGHT_COLOR) if history == "ancestral" else (EVOLVED_COLOR, EVOLVED_LIGHT_COLOR)
        axD.barh(y, row.rate_R2_contribution, height=bar_h, color=dark, edgecolor="white", lw=0.5, zorder=3)
        axD.barh(y, row.lag_R2_contribution, left=row.rate_R2_contribution, height=bar_h, color=light, edgecolor="white", lw=0.5, zorder=3)
        y_ticks.append(y); y_labels.append(hlabel)
axD.set_yticks(y_ticks)
axD.set_yticklabels(y_labels, fontsize=5.0)
for tick, (history, _) in zip(axD.get_yticklabels(), [h for _ in TRAIT_DOSES for h in histories]):
    tick.set_color(ANCESTRAL_COLOR if history == "ancestral" else EVOLVED_COLOR)
axD.set_ylim(len(TRAIT_DOSES) - 0.5, -0.5)
axD.set_xlim(0, float(table_s14.joint_R2.max()) * 1.26)
axD.set_xlabel("Variance in competitive\nposition explained ($R^2$)")
axD.tick_params(axis="x", labelsize=5.7); axD.tick_params(axis="y", length=0)
axD.grid(axis="x", color="0.88", lw=0.5, zorder=0); axD.set_axisbelow(True)
for side in ["top", "right", "left"]:
    axD.spines[side].set_visible(False)
for di, dose in enumerate(TRAIT_DOSES):
    axD.text(0.004, di - 0.355, f"{dose:g} µg ml$^{{-1}}$ streptomycin", ha="left", va="bottom", fontsize=5.0, color="0.30", zorder=4)
axD.legend(handles=[(Patch(facecolor=ANCESTRAL_COLOR, edgecolor="none"), Patch(facecolor=EVOLVED_COLOR, edgecolor="none")),
                    (Patch(facecolor=ANCESTRAL_LIGHT_COLOR, edgecolor="none"), Patch(facecolor=EVOLVED_LIGHT_COLOR, edgecolor="none"))],
           labels=["Maximum growth rate", "Lag time"], handler_map={tuple: HandlerTuple(ndivide=None, pad=0)},
           loc="lower right", bbox_to_anchor=(1.0, -0.02), frameon=False, fontsize=4.8, handlelength=1.0, handletextpad=0.4)
fig.subplots_adjust(top=0.90, bottom=0.25, left=0.07, right=0.99)
save_figure(fig, "Figure_5_resistance_filtering_and_growth_traits")

## Figure S10: monoculture growth-trait landscapes
Maximum growth rate and lag time are not shown at 200 ug/ml for ancestral *A. caviae* and
*P. chlororaphis*, which barely grow there.

In [6]:
trait_labels = {"max_growth_rate": r"Maximum growth rate (OD$_{600}$ min$^{-1}$)", "yield": r"Yield (OD$_{600}$)",
                "auc": r"Growth-curve AUC (OD$_{600}$ h)", "lag_time_h": "Lag time (h)"}
KINETIC_TRAITS = {"max_growth_rate", "lag_time_h"}
strains = ["HAMBI_1287", "HAMBI_1972_ANC", "HAMBI_1972_EVO", "HAMBI_1977"]
fig = plt.figure(figsize=(183 * MM, 218 * MM))
gs = fig.add_gridspec(len(GROWTH_TRAITS), 5, width_ratios=[1, 1, 1, 1, 0.055], hspace=0.20, wspace=0.13)
for row, trait in enumerate(GROWTH_TRAITS):
    grids = {}
    for sp in strains:
        grid = (growth_means.loc[growth_means.strain == sp].pivot(index="carbon_g_L", columns="streptomycin", values=trait)
                .reindex(index=CARBON_VALUES, columns=STREPTOMYCIN_VALUES).to_numpy(float))
        if trait in KINETIC_TRAITS and sp in {"HAMBI_1972_ANC", "HAMBI_1977"}:
            grid[:, list(STREPTOMYCIN_VALUES).index(200.0)] = np.nan
        grids[sp] = grid
    values = np.concatenate([g[np.isfinite(g)] for g in grids.values()])
    for col, sp in enumerate(strains):
        ax = fig.add_subplot(gs[row, col])
        im = ax.imshow(grids[sp], origin="upper", aspect="auto", cmap=CMAP_SEQ, vmin=values.min(), vmax=values.max())
        format_grid_axis(ax, show_x=(row == len(GROWTH_TRAITS) - 1), show_y=(col == 0), ylabel="")
        ax.tick_params(axis="y", labelsize=4.8, pad=0.8)
        if row == 0:
            ax.text(0.5, 1.025, strain_plot_label(GROWTH_STRAIN_LABELS[sp]), transform=ax.transAxes, ha="center", va="bottom", fontsize=5.8)
        if col == 0:
            ax.text(-0.29, 0.5, trait_labels[trait], transform=ax.transAxes, rotation=90, va="center", ha="center", fontsize=5.25)
    fig.colorbar(im, cax=fig.add_subplot(gs[row, 4])).ax.tick_params(labelsize=5.1)
fig.supylabel(CARBON_AXIS_LABEL, x=0.021, fontsize=6.6)
fig.subplots_adjust(top=0.94, bottom=0.07, left=0.14, right=0.96)
save_figure(fig, "Figure_S10_growth_trait_landscapes")

## Figure S11: dose-resolved partial associations
Lines are mean partial slopes at each dose (Table S13); bands are bootstrap 95% CIs from
resampling the eight landscapes.

In [7]:
fig, axes = plt.subplots(2, 3, figsize=(178 * MM, 118 * MM), sharey="row", squeeze=False)
rng = np.random.default_rng(RANDOM_SEED + 8800)
for row, (trait, slope_col, xlabel) in enumerate([("lag_time_h", "lag_slope", "Relative lag time (s.d.)"),
                                                  ("max_growth_rate", "rate_slope", "Relative maximum growth rate (s.d.)")]):
    xcol = f"z_relative_{trait}"
    for col, dose in enumerate(TRAIT_DOSES):
        ax = axes[row, col]
        ax.axhline(0, color="0.82", lw=0.6); ax.axvline(0, color="0.82", lw=0.6)
        for community, history, color in [("ANC_THREE", "ancestral", ANCESTRAL_COLOR), ("EVO_THREE", "resistance-evolved", EVOLVED_COLOR)]:
            dat = trait_condition_plot.loc[(trait_condition_plot.community == community) & (trait_condition_plot.streptomycin == dose)]
            ax.scatter(dat[xcol], dat["q_endpoint"], s=8, facecolor="white", edgecolor=color, lw=0.45, alpha=0.45, rasterized=True)
            slopes = dose_slopes.loc[(dose_slopes.history == history) & (dose_slopes.streptomycin == dose), slope_col].to_numpy(float)
            xx = np.linspace(dat[xcol].min(), dat[xcol].max(), 120)
            boot = np.median(slopes[rng.integers(0, len(slopes), size=(10000, len(slopes)))], axis=1)
            lo, hi = np.quantile(boot[:, None] * xx[None, :], [0.025, 0.975], axis=0)
            ax.fill_between(xx, lo, hi, color=color, alpha=0.30, linewidth=0, zorder=3)
            ax.plot(xx, np.median(slopes) * xx, color=color, lw=0.9, zorder=4)
        if row == 0:
            ax.set_title(f"{dose:g} µg ml$^{{-1}}$ streptomycin", fontsize=6.5)
        ax.set_xlabel(xlabel, fontsize=6.2); ax.tick_params(labelsize=5.4)
        if col == 0:
            ax.set_ylabel(r"Competitive position $q_i$", fontsize=6.3)
fig.subplots_adjust(left=0.09, right=0.99, bottom=0.10, top=0.93, hspace=0.36, wspace=0.22)
save_figure(fig, "Figure_S11_growth_traits_vs_competitive_position")

## Figure S12: growth of ancestral and resistance-evolved *A. caviae*
Monoculture replicates averaged across carbon within each dose; maximum growth rate and lag
time are not shown for the ancestral strain at 200 ug/ml.

In [8]:
fig, axes = plt.subplots(2, 2, figsize=(145 * MM, 126 * MM), sharex=True)
for i, (ax, trait) in enumerate(zip(axes.ravel(), GROWTH_TRAITS)):
    for strain, color in [("HAMBI_1972_ANC", ANCESTRAL_COLOR), ("HAMBI_1972_EVO", EVOLVED_COLOR)]:
        sub = growth.loc[growth.strain == strain, ["replicate", "streptomycin", trait]].copy()
        if trait in KINETIC_TRAITS and strain == "HAMBI_1972_ANC":
            sub.loc[sub.streptomycin == 200.0, trait] = np.nan
        sub = sub.groupby(["replicate", "streptomycin"])[trait].mean().reset_index()
        for xi, dose in enumerate(STREPTOMYCIN_VALUES):
            vals = sub.loc[sub.streptomycin == dose, trait].dropna().to_numpy(float)
            if len(vals):
                replicate_mean_ci(ax, xi + (-0.06 if strain == "HAMBI_1972_ANC" else 0.06), vals, color,
                                  seed=7000 + i * 100 + xi + (0 if strain == "HAMBI_1972_ANC" else 50), jitter_width=0.018, size=10)
    ax.set_xticks(range(4)); ax.set_xticklabels([f"{x:g}" for x in STREPTOMYCIN_VALUES])
    ax.set_ylabel(trait_labels[trait], fontsize=6.5)
    if i >= 2:
        ax.set_xlabel(STREP_AXIS_LABEL)
fig.legend(handles=[Line2D([0], [0], marker="o", color=ANCESTRAL_COLOR, lw=0, label=r"Ancestral $\it{Aeromonas\ caviae}$"),
                    Line2D([0], [0], marker="o", color=EVOLVED_COLOR, lw=0, label=r"Resistance-evolved $\it{Aeromonas\ caviae}$")],
           loc="lower center", bbox_to_anchor=(0.5, 0.005), ncol=2, frameon=False, fontsize=5.4)
fig.subplots_adjust(bottom=0.14, top=0.97, hspace=0.30, wspace=0.33)
save_figure(fig, "Figure_S12_focal_growth_traits")